# CrewAI AI Radar Verifier - Draft v1.0

A text-first, portfolio-ready workflow for extracting and verifying factual claims from AI Radar intelligence briefs using CrewAI, Claude via Anthropic, and free DuckDuckGo search.

## 1. Introduction

This notebook ingests a `.txt` AI Radar brief, extracts discrete factual claims, researches each claim online, verifies the claims against inspected source content, and exports an audit-ready table with supporting URLs.

Version one is text-first. OCR and image/PDF extraction are intentionally out of scope.

## 2. Architecture Overview

The workflow is organized around seven CrewAI roles:

1. Document Intake Agent
2. Claim Extraction Agent
3. Search Strategy Agent
4. Research Agent
5. Verification Agent
6. Source Ranking Agent
7. Report Agent

Reusable modules under `src/` perform the operational work so the notebook stays readable and auditable.

## 3. Environment Setup

Install dependencies with `pip install -r requirements.txt`, then create a `.env` file from `.env.example`.

Required environment variable:

- `ANTHROPIC_API_KEY`

DuckDuckGo search is free and does not require an API key.

## 4. Imports

In [20]:
from __future__ import annotations

import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from crewai import Agent, Crew, Process, Task

from src.claim_extraction import ClaudeClaimExtractor, claims_as_dicts, fallback_extract_claims
from src.export import export_results, results_to_dataframe
from src.intake import load_brief
from src.search_provider import DuckDuckGoSearchProvider
from src.verification import ClaudeVerifier, build_search_queries, verify_claims

## 5. Configuration

In [21]:
load_dotenv()

BRIEF_PATH = Path("sample_briefs/elevance_health_ai_brief.txt")
OUTPUT_DIR = Path("outputs")
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter Anthropic API key: ").strip()
else:
    os.environ["ANTHROPIC_API_KEY"] = os.environ["ANTHROPIC_API_KEY"].strip()

MODEL = os.getenv("ANTHROPIC_MODEL", "").strip() or None
CREWAI_MODEL = MODEL or "claude-3-5-haiku-20241022"
MAX_CLAIMS = int(os.getenv("AIRADAR_MAX_CLAIMS", "8"))
DUCKDUCKGO_MAX_RESULTS = int(os.getenv("DUCKDUCKGO_MAX_RESULTS", "5"))

missing = [name for name in ["ANTHROPIC_API_KEY"] if not os.getenv(name)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing)}")

print(f"Brief: {BRIEF_PATH}")
print(f"Claude model override: {MODEL or 'auto fallback'}")
print(f"CrewAI definition model: {CREWAI_MODEL}")
print(f"Max claims: {MAX_CLAIMS}")
print(f"DuckDuckGo max results per query: {DUCKDUCKGO_MAX_RESULTS}")

Available Claude models found: claude-opus-4-8, claude-opus-4-7, claude-sonnet-4-6, claude-opus-4-6, claude-opus-4-5-20251101, claude-haiku-4-5-20251001
Brief: sample_briefs\elevance_health_ai_brief.txt
Claude model: claude-opus-4-8
Max claims: 8
DuckDuckGo max results per query: 5


## 6. Text Brief Input

In [22]:
brief = load_brief(BRIEF_PATH)

print(f"Title: {brief.title}")
print(f"Company: {brief.company}")
print(f"Date: {brief.date}")
print(f"Characters: {len(brief.raw_text):,}")

Title: AI Radar Brief: Elevance Health
Company: Elevance Health
Date: 2026-05-28
Characters: 1,526


## 7. Document Parsing

In [23]:
section_summary = pd.DataFrame(
    [{"section": name, "characters": len(text), "preview": text[:180]} for name, text in brief.sections.items()]
)
display(section_summary)

,section,characters,preview
0,AI Radar Brief: Elevance Health,312,Company: Elevance Health\nDate: 2026-05-28\nEd...
1,KPI Metric Cards,289,- Elevance Health reported 2024 operating reve...
2,Enterprise AI and Digital Operations,165,Elevance Health has invested in digital tools ...
3,Carelon Strategy,166,Carelon is positioned as Elevance Health's hea...
4,Governance and Risk,187,The company faces ongoing regulatory scrutiny ...
5,Update Notes,214,"- Verify membership, revenue, and business-seg..."


## 8. Claim Extraction

In [24]:
claim_extractor = ClaudeClaimExtractor(model=MODEL)

try:
    claims = claim_extractor.extract(brief.raw_text, brief.sections, max_claims=MAX_CLAIMS)
except Exception as exc:
    print(f"Claude extraction failed; using deterministic text fallback. Reason: {exc}")
    claims = fallback_extract_claims(brief.sections, max_claims=MAX_CLAIMS)

claims_df = pd.DataFrame(claims_as_dicts(claims))
display(claims_df)

Claude extraction failed; using deterministic text fallback. Reason: RetryError[<Future at 0x24cdcfd1610 state=finished raised BadRequestError>]


,claim_id,brief_section,claim,claim_type,entities,date_reference,search_queries
0,C001,KPI Metric Cards,Elevance Health reported 2024 operating revenu...,other,[],None,[Elevance Health reported 2024 operating reven...
1,C002,KPI Metric Cards,The company served more than 45 million medica...,other,[],None,[The company served more than 45 million medic...
2,C003,Enterprise AI and Digital Operations,Elevance Health has invested in digital tools ...,other,[],None,[Elevance Health has invested in digital tools...
3,C004,Carelon Strategy,Carelon is positioned as Elevance Health's hea...,other,[],None,[Carelon is positioned as Elevance Health's he...
4,C005,Governance and Risk,The company faces ongoing regulatory scrutiny ...,other,[],None,[The company faces ongoing regulatory scrutiny...
5,C006,Update Notes,"Verify membership, revenue, and business-segme...",other,[],None,"[Verify membership, revenue, and business-segm..."


## 9. CrewAI Agent Definitions

In [25]:
llm_config = f"anthropic/{CREWAI_MODEL}"

document_intake_agent = Agent(
    role="Document Intake Agent",
    goal="Parse text-based AI Radar briefs and identify the company, date, sections, and source context.",
    backstory="A meticulous intelligence operations analyst who prepares briefs for downstream verification.",
    llm=llm_config,
    verbose=True,
)

claim_extraction_agent = Agent(
    role="Claim Extraction Agent",
    goal="Extract discrete factual claims that can be verified from public evidence.",
    backstory="A fact-checking editor who separates claims from narrative and opinion.",
    llm=llm_config,
    verbose=True,
)

search_strategy_agent = Agent(
    role="Search Strategy Agent",
    goal="Generate source-seeking queries that prioritize primary evidence.",
    backstory="A research strategist skilled at turning claims into precise evidence queries.",
    llm=llm_config,
    verbose=True,
)

research_agent = Agent(
    role="Research Agent",
    goal="Use DuckDuckGo search results and inspected pages to collect relevant evidence.",
    backstory="A web researcher who avoids snippet-only conclusions and keeps URLs traceable.",
    llm=llm_config,
    verbose=True,
)

verification_agent = Agent(
    role="Verification Agent",
    goal="Assign conservative verification statuses based only on inspected evidence.",
    backstory="An evidence auditor who marks weak support as Not Found rather than overclaiming.",
    llm=llm_config,
    verbose=True,
)

source_ranking_agent = Agent(
    role="Source Ranking Agent",
    goal="Rank company, regulatory, filing, and other primary sources ahead of secondary coverage.",
    backstory="A source-quality specialist focused on audit defensibility.",
    llm=llm_config,
    verbose=True,
)

report_agent = Agent(
    role="Report Agent",
    goal="Create the final audit-ready table with claim status, confidence, evidence notes, and URLs.",
    backstory="A portfolio analyst who turns verification work into clean decision-ready reporting.",
    llm=llm_config,
    verbose=True,
)

agents = [
    document_intake_agent,
    claim_extraction_agent,
    search_strategy_agent,
    research_agent,
    verification_agent,
    source_ranking_agent,
    report_agent,
]

print(f"Defined {len(agents)} CrewAI agents.")

Defined 7 CrewAI agents.


## 10. CrewAI Task Definitions

In [26]:
intake_task = Task(
    description="Review the loaded brief metadata and confirm the text-first verification scope.",
    expected_output="A concise intake summary with title, company, date, and detected sections.",
    agent=document_intake_agent,
)

claim_task = Task(
    description="Review the extracted claims and identify which are highest value for evidence-backed verification.",
    expected_output="A prioritized claim list with section, claim type, entities, and date references.",
    agent=claim_extraction_agent,
)

search_task = Task(
    description="Assess the generated search queries and improve primary-source targeting where needed.",
    expected_output="Search strategy notes that prioritize SEC filings, company releases, annual reports, and official pages.",
    agent=search_strategy_agent,
)

research_task = Task(
    description="Evaluate whether retrieved evidence is sufficient and traceable for each claim.",
    expected_output="Research notes identifying the strongest inspected sources per claim.",
    agent=research_agent,
)

verification_task = Task(
    description="Check verification decisions for conservatism and consistency with the allowed status labels.",
    expected_output="A quality-control summary of Confirmed, Partially Confirmed, Contradicted, and Not Found decisions.",
    agent=verification_agent,
)

ranking_task = Task(
    description="Review source ordering and confirm that primary sources are favored over secondary sources.",
    expected_output="Source ranking notes and any warnings about weak or inaccessible evidence.",
    agent=source_ranking_agent,
)

report_task = Task(
    description="Summarize the final report fields and portfolio value of the verification workflow.",
    expected_output="A short report narrative explaining the output table, limitations, and audit trail.",
    agent=report_agent,
)

tasks = [intake_task, claim_task, search_task, research_task, verification_task, ranking_task, report_task]
crew = Crew(agents=agents, tasks=tasks, process=Process.sequential, verbose=True)

print(f"Defined {len(tasks)} CrewAI tasks.")

Defined 7 CrewAI tasks.


## 11. Crew Execution

The cells below execute the verification workflow using the reusable modules. A lightweight CrewAI quality-control kickoff can be enabled after the evidence table is generated.

In [27]:
search_provider = DuckDuckGoSearchProvider(max_results=DUCKDUCKGO_MAX_RESULTS)
verifier = ClaudeVerifier(model=MODEL)

query_preview = []
for claim in claims:
    query_preview.append(
        {
            "claim_id": claim.claim_id,
            "claim": claim.claim,
            "queries": build_search_queries(claim, company=brief.company),
        }
    )

display(pd.DataFrame(query_preview))

,claim_id,claim,queries
0,C001,Elevance Health reported 2024 operating revenu...,[Elevance Health reported 2024 operating reven...
1,C002,The company served more than 45 million medica...,[The company served more than 45 million medic...
2,C003,Elevance Health has invested in digital tools ...,[Elevance Health has invested in digital tools...
3,C004,Carelon is positioned as Elevance Health's hea...,[Carelon is positioned as Elevance Health's he...
4,C005,The company faces ongoing regulatory scrutiny ...,[The company faces ongoing regulatory scrutiny...
5,C006,"Verify membership, revenue, and business-segme...","[Verify membership, revenue, and business-segm..."


In [28]:
verification_results = verify_claims(
    claims=claims,
    search_provider=search_provider,
    verifier=verifier,
    company=brief.company,
)

results_df = results_to_dataframe(verification_results)
display(results_df)

RetryError: RetryError[<Future at 0x24cda459550 state=finished raised BadRequestError>]

In [ ]:
RUN_CREWAI_QA = False

if RUN_CREWAI_QA:
    crew_inputs = {
        "brief_title": brief.title,
        "company": brief.company,
        "claims": claims_df.to_dict(orient="records"),
        "verification_results": results_df.to_dict(orient="records"),
    }
    crew_result = crew.kickoff(inputs=crew_inputs)
    print(crew_result)
else:
    print("CrewAI QA kickoff is defined but skipped. Set RUN_CREWAI_QA = True to run the agent review pass.")

## 12. Verification Results

In [ ]:
status_summary = results_df.groupby("Verification Status", dropna=False).size().reset_index(name="Claims")
display(status_summary)
display(results_df)

## 13. Export Results

In [ ]:
export_paths = export_results(verification_results, output_dir=OUTPUT_DIR)
for kind, path in export_paths.items():
    print(f"{kind.upper()}: {path}")

## 14. Limitations

- This version accepts `.txt` briefs only.
- OCR, scanned PDFs, and image extraction are not implemented in v1.
- Search coverage depends on DuckDuckGo results, source availability, and website access controls.
- The verifier marks weak or inaccessible evidence as `Not Found` instead of filling gaps with assumptions.
- Some claims may require paid filings databases, transcript providers, or analyst reports that are outside the public web workflow.

## 15. How This Demonstrates Agentic AI Experience

This notebook demonstrates a practical agentic AI workflow: CrewAI roles define the collaboration model, Claude extracts and evaluates claims, DuckDuckGo supplies free search coverage, source ranking favors audit-grade evidence, and the final report preserves traceable URLs and conservative verification decisions. The result is a reusable verifier that is more substantial than a toy demo while remaining clear enough for portfolio review.